In [ ]:
!ls "/content/drive/MyDrive/oral_project"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
!find /content/drive/MyDrive -maxdepth 3 -iname "*.zip"

In [ ]:
!unzip -q "/content/drive/MyDrive/NDB-UFES An oral cancer and leukoplakia dataset composed of histopathological images and patient data.zip" -d /content/data
!ls /content/data

In [ ]:
!find /content/data -maxdepth 2 | head -50

In [ ]:
import os, pandas as pd

DATA = "/content/data/NDB-UFES An oral cancer and leukoplakia dataset composed of histopathological images and patient data"

!unzip -q "{DATA}/images.zip" -d "{DATA}/images_extracted"
print(os.listdir(DATA))

In [ ]:
df = pd.read_csv(f"{DATA}/ndb-ufes.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
print(df['diagnosis'].value_counts(), "\n")
print(df['dysplasia_severity'].value_counts(dropna=False), "\n")

for c in ['TaskII', 'TaskIII', 'TaskIV']:
    print(c)
    print(df[c].value_counts(), "\n")

print("images:", len(df),
      "| lesions:", df['lesion_id'].nunique(),
      "| patients:", df['patient_id'].nunique())
print(df.groupby('patient_id').size().describe())

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

files = [f for f in glob.glob(f"{DATA}/images_extracted/**/*", recursive=True)
         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print("Image files found:", len(files))
path_map = {os.path.basename(f): f for f in files}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, label in enumerate(['OSCC', 'Leukoplakia']):
    samples = df[df['TaskII'] == label].sample(3, random_state=0)
    for col, (_, r) in enumerate(samples.iterrows()):
        img = Image.open(path_map[r['path']])
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{label} | patient {r['patient_id']}")
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()
print("Image size (width, height):", img.size)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# 1. Does any patient have BOTH classes? (this would be unusual)
mix = df.groupby('patient_id')['TaskII'].nunique()
print("Patients with both classes:", (mix > 1).sum())

# 2. Make the 5 folds
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df['fold'] = -1
for fold, (train_idx, test_idx) in enumerate(
        sgkf.split(df, y=df['TaskII'], groups=df['patient_id'])):
    df.loc[test_idx, 'fold'] = fold

# 3. Check: no patient appears in two folds
print("Patients in more than one fold:",
      (df.groupby('patient_id')['fold'].nunique() > 1).sum())

# 4. Check: size and class balance of each fold
print(df.groupby(['fold', 'TaskII']).size().unstack())
print(df.groupby('fold')['patient_id'].nunique())

# 5. Save so we reuse exactly the same split later
df.to_csv("/content/drive/MyDrive/oral_project/folds.csv", index=False)

In [ ]:
os.makedirs("/content/drive/MyDrive/oral_project", exist_ok=True)
df.to_csv("/content/drive/MyDrive/oral_project/folds.csv", index=False)
print("Saved:", os.path.exists("/content/drive/MyDrive/oral_project/folds.csv"))

In [ ]:
print(os.path.exists("/content/drive/MyDrive/oral_project/folds.csv"))

In [ ]:
patch_dir = f"{DATA}/patch"
print(os.listdir(patch_dir)[:10])

pfiles = [f for f in glob.glob(f"{patch_dir}/**/*", recursive=True)
          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print("Patch files found:", len(pfiles))
print(pfiles[:3])

pdirs = [d for d in glob.glob(f"{patch_dir}/**/", recursive=True)]
print("Subfolders:", len(pdirs))

In [ ]:
patch_dir = f"{DATA}/patch"
!unzip -q "{patch_dir}/images-patches.zip" -d "{patch_dir}/extracted"

pfiles = [f for f in glob.glob(f"{patch_dir}/extracted/**/*", recursive=True)
          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print("Patch files found:", len(pfiles))
print(pfiles[:3])

pdf = pd.read_csv(f"{patch_dir}/sabpatch_parsed_folders.csv")
print(pdf.shape)
print(pdf.columns.tolist())
print(pdf.head(3).to_string())

In [ ]:
!unzip -l "{patch_dir}/images-patches.zip" | head -15
!ls -la "{patch_dir}/extracted" | head

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

W, H = 512, 384
def load_resized(fname):
    img = Image.open(path_map[fname]).convert("RGB").resize((W, H), Image.LANCZOS)
    return np.array(img)

images = {f: load_resized(f) for f in df['path']}
print(len(images), images[df['path'].iloc[0]].shape)

In [ ]:
class OralDataset(Dataset):
    def __init__(self, dataframe, train, crop=224):
        self.df = dataframe.reset_index(drop=True)
        self.train, self.crop = train, crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = images[row['path']]
        h, w, _ = img.shape
        if self.train:   # random crop
            y = np.random.randint(0, h - self.crop + 1)
            x = np.random.randint(0, w - self.crop + 1)
        else:            # centre crop
            y, x = (h - self.crop) // 2, (w - self.crop) // 2
        patch = img[y:y+self.crop, x:x+self.crop]
        t = torch.from_numpy(patch).permute(2, 0, 1).float() / 255.0
        t = (t - 0.5) / 0.5
        label = 1 if row['TaskII'] == 'OSCC' else 0
        return t, label

test_fold = 0
train_df = df[df['fold'] != test_fold]
test_df  = df[df['fold'] == test_fold]
print("Patients in both train and test:",
      set(train_df['patient_id']) & set(test_df['patient_id']))

train_loader = DataLoader(OralDataset(train_df, True), batch_size=16, shuffle=True)
xb, yb = next(iter(train_loader))
print(xb.shape, yb)

In [ ]:
import torch.nn as nn

def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1),
        nn.BatchNorm2d(c_out),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            block(3, 16), block(16, 32), block(32, 64), block(64, 128)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(self.drop(x))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN().to(device)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

out = model(xb.to(device))
print(out.shape)

In [ ]:
import random
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def five_crops(img, c=224):
    h, w, _ = img.shape
    pos = [((h-c)//2, (w-c)//2), (0, 0), (0, w-c), (h-c, 0), (h-c, w-c)]
    return [img[y:y+c, x:x+c] for y, x in pos]

def to_tensor(p):
    t = torch.from_numpy(np.ascontiguousarray(p)).permute(2, 0, 1).float() / 255.0
    return (t - 0.5) / 0.5

@torch.no_grad()
def predict_probs(model, dataframe):
    model.eval()
    probs = []
    for fname in dataframe['path']:
        batch = torch.stack([to_tensor(p) for p in five_crops(images[fname])]).to(device)
        probs.append(torch.softmax(model(batch), dim=1)[:, 1].mean().item())
    return np.array(probs)

def metrics(y_true, probs, thr=0.5):
    y_pred = (probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(sens=tp/(tp+fn), spec=tn/(tn+fp),
                macro_f1=f1_score(y_true, y_pred, average='macro'),
                auc=roc_auc_score(y_true, probs),
                tn=tn, fp=fp, fn=fn, tp=tp)

In [ ]:
def train_fold(test_fold, epochs=30, seed=0, lr=1e-3):
    set_seed(seed)
    train_df = df[df['fold'] != test_fold]
    test_df  = df[df['fold'] == test_fold]

    y_train = (train_df['TaskII'] == 'OSCC').astype(int).values
    counts = np.bincount(y_train)
    weights = torch.tensor(len(y_train) / (2 * counts), dtype=torch.float).to(device)

    loader = DataLoader(OralDataset(pd.concat([train_df] * 4), True),
                        batch_size=16, shuffle=True)
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    for ep in range(epochs):
        model.train()
        total = 0
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(device), yb_.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb_), yb_)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb_)
        if (ep + 1) % 5 == 0:
            print(f"epoch {ep+1}: train loss {total / len(loader.dataset):.3f}")

    y_test = (test_df['TaskII'] == 'OSCC').astype(int).values
    return model, metrics(y_test, predict_probs(model, test_df))

model0, res0 = train_fold(test_fold=0)
print(res0)

In [ ]:
all_res = {0: res0}
fold_models = {0: model0}
for f in [1, 2, 3, 4]:
    print(f"--- test fold {f} ---")
    fold_models[f], all_res[f] = train_fold(test_fold=f)

res_df = pd.DataFrame(all_res).T[['sens', 'spec', 'macro_f1', 'auc']].astype(float)
print(res_df.round(3))
print("\nMean:\n", res_df.mean().round(3))
print("\nStd:\n", res_df.std().round(3))

os.makedirs("/content/drive/MyDrive/oral_project", exist_ok=True)
res_df.to_csv("/content/drive/MyDrive/oral_project/baseline_cv_results.csv")
for f, m in fold_models.items():
    torch.save(m.state_dict(), f"/content/drive/MyDrive/oral_project/baseline_fold{f}.pt")

In [ ]:
from PIL import ImageFilter
import io

LEVELS = {
    'blur':  [1.0, 2.0, 4.0],
    'noise': [10, 20, 40],
    'jpeg':  [40, 20, 8],
    'dark':  [0.7, 0.5, 0.3],
}

def degrade(img, kind, level, seed=0):
    if level == 0:
        return img
    p = LEVELS[kind][level - 1]
    if kind == 'blur':
        return np.array(Image.fromarray(img).filter(ImageFilter.GaussianBlur(p)))
    if kind == 'noise':
        rng = np.random.default_rng(seed)
        out = img.astype(np.float32) + rng.normal(0, p, img.shape)
        return np.clip(out, 0, 255).astype(np.uint8)
    if kind == 'jpeg':
        buf = io.BytesIO()
        Image.fromarray(img).save(buf, format='JPEG', quality=p)
        buf.seek(0)
        return np.array(Image.open(buf).convert('RGB'))
    if kind == 'dark':
        return np.clip(img.astype(np.float32) * p, 0, 255).astype(np.uint8)

fname = df[df['TaskII'] == 'OSCC']['path'].iloc[0]
base = images[fname]
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for r, kind in enumerate(LEVELS):
    for lvl in range(4):
        d = degrade(base, kind, lvl)[64:320, 128:384]
        axes[r, lvl].imshow(d)
        axes[r, lvl].set_title(f"{kind} level {lvl}")
        axes[r, lvl].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
LEVELS = {
    'blur':  [1.0, 2.0, 3.0],
    'noise': [8, 16, 25],
    'jpeg':  [40, 20, 8],
    'dark':  [0.7, 0.5, 0.35],
}

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for r, kind in enumerate(['blur', 'noise', 'dark']):
    for lvl in range(4):
        d = degrade(base, kind, lvl)[64:320, 128:384]
        axes[r, lvl].imshow(d)
        axes[r, lvl].set_title(f"{kind} level {lvl}")
        axes[r, lvl].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict_probs_deg(model, dataframe, kind, level):
    model.eval()
    probs = []
    for i, fname in enumerate(dataframe['path']):
        img = degrade(images[fname], kind, level, seed=i)
        batch = torch.stack([to_tensor(p) for p in five_crops(img)]).to(device)
        probs.append(torch.softmax(model(batch), dim=1)[:, 1].mean().item())
    return np.array(probs)

rows = []
for kind in LEVELS:
    for level in range(4):
        per_fold = []
        for f in range(5):
            test_df = df[df['fold'] == f]
            y = (test_df['TaskII'] == 'OSCC').astype(int).values
            p = predict_probs_deg(fold_models[f], test_df, kind, level)
            per_fold.append(metrics(y, p))
        m = pd.DataFrame(per_fold)[['sens', 'spec', 'macro_f1', 'auc']].astype(float)
        rows.append(dict(kind=kind, level=level, **m.mean().round(3).to_dict()))

rob = pd.DataFrame(rows)
print(rob.to_string(index=False))
rob.to_csv("/content/drive/MyDrive/oral_project/baseline_robustness.csv", index=False)

In [ ]:
class AugDataset(OralDataset):
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = images[row['path']]
        if np.random.rand() < 0.5:
            kind = random.choice(list(LEVELS))
            level = np.random.randint(1, 4)
            img = degrade(img, kind, level, seed=np.random.randint(1_000_000))
        h, w, _ = img.shape
        y = np.random.randint(0, h - self.crop + 1)
        x = np.random.randint(0, w - self.crop + 1)
        label = 1 if row['TaskII'] == 'OSCC' else 0
        return to_tensor(img[y:y+self.crop, x:x+self.crop]), label

def train_fold(test_fold, epochs=30, seed=0, lr=1e-3, ds_class=OralDataset):
    set_seed(seed)
    train_df = df[df['fold'] != test_fold]
    test_df  = df[df['fold'] == test_fold]
    y_train = (train_df['TaskII'] == 'OSCC').astype(int).values
    counts = np.bincount(y_train)
    weights = torch.tensor(len(y_train) / (2 * counts), dtype=torch.float).to(device)
    loader = DataLoader(ds_class(pd.concat([train_df] * 4), True),
                        batch_size=16, shuffle=True)
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    for ep in range(epochs):
        model.train()
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(device), yb_.to(device)
            opt.zero_grad()
            loss_fn(model(xb_), yb_).backward()
            opt.step()
    y_test = (test_df['TaskII'] == 'OSCC').astype(int).values
    return model, metrics(y_test, predict_probs(model, test_df))

aug_models = {}
for f in range(5):
    print(f"training aug model, test fold {f}")
    aug_models[f], _ = train_fold(f, ds_class=AugDataset)
    torch.save(aug_models[f].state_dict(),
               f"/content/drive/MyDrive/oral_project/aug_fold{f}.pt")
print("done")

In [ ]:
def robustness_table(models):
    rows = []
    for kind in LEVELS:
        for level in range(4):
            per_fold = []
            for f in range(5):
                test_df = df[df['fold'] == f]
                y = (test_df['TaskII'] == 'OSCC').astype(int).values
                p = predict_probs_deg(models[f], test_df, kind, level)
                per_fold.append(metrics(y, p))
            m = pd.DataFrame(per_fold)[['sens', 'spec', 'macro_f1', 'auc']].astype(float)
            rows.append(dict(kind=kind, level=level, **m.mean().round(3).to_dict()))
    return pd.DataFrame(rows)

aug_rob = robustness_table(aug_models)
aug_rob.to_csv("/content/drive/MyDrive/oral_project/aug_robustness.csv", index=False)

cmp = rob.merge(aug_rob, on=['kind', 'level'], suffixes=('_base', '_aug'))
print(cmp[['kind', 'level', 'sens_base', 'sens_aug', 'auc_base', 'auc_aug']].to_string(index=False))

In [ ]:
import pickle

models = {'base': {0: fold_models}, 'aug': {0: aug_models}}
for seed in [1, 2]:
    for name, ds in [('base', OralDataset), ('aug', AugDataset)]:
        models[name][seed] = {}
        for f in range(5):
            print(name, "seed", seed, "fold", f)
            models[name][seed][f], _ = train_fold(f, seed=seed, ds_class=ds)

CONDS = [('clean', 0)] + [(k, l) for k in LEVELS for l in (1, 2, 3)]
probs = {}
for name in models:
    for kind, level in CONDS:
        arr = np.zeros((len(models[name]), len(df)))
        for si, seed in enumerate(models[name]):
            for f in range(5):
                idx = df.index[df['fold'] == f]
                k = 'blur' if kind == 'clean' else kind
                arr[si, idx] = predict_probs_deg(models[name][seed][f], df.loc[idx], k, level)
        probs[(name, kind, level)] = arr

pickle.dump(probs, open("/content/drive/MyDrive/oral_project/probs.pkl", "wb"))

y_all = (df['TaskII'] == 'OSCC').astype(int).values
for name in models:
    print(name, "clean pooled AUC:",
          round(roc_auc_score(y_all, probs[(name, 'clean', 0)].mean(0)), 3))